In [1]:
#importing libraries
import gradio as gr
import pandas as pd
import numpy as np
import faiss
import json
import os
import csv

In [2]:
import shutil
from sentence_transformers import SentenceTransformer

W0714 21:00:45.962000 9848 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
from llama_cpp import Llama

In [4]:
from sklearn.metrics.pairwise import cosine_similarity
import re
from datetime import datetime
from paddleocr import PaddleOCR
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from textwrap import wrap
from reportlab.lib.utils import ImageReader

In [5]:
# === paths of all files ===
base_dir = r"C:\Users\hp\Downloads\Project folder of Chatter"
faiss_path = os.path.join(base_dir, "faiss_index_PaddleOCR.index")
csv_path = os.path.join(base_dir, "extracted_text_PaddleOCR2/cleaned_text_PaddleOCR2/chunked_text_PaddleOCR.csv")
text_folder = os.path.join(base_dir, "extracted_text_PaddleOCR2/cleaned_text_PaddleOCR2")
logo_path = os.path.join(base_dir, "Logo.png")
mistral_path = r"C:\Users\hp\Downloads\mistral-7b-instruct-v0.1.Q4_K_M.gguf"
CACHE_FILE = "cache.json"
FEEDBACK_FILE = "feedback_log.csv"

In [ ]:
# === Load all components ===

# Load the FAISS index from disk — used for efficient similarity search over embedded text chunks
index = faiss.read_index(faiss_path)

# Load the preprocessed and chunked text data (from OCR'd documents) into a DataFrame
df_chunks = pd.read_csv(csv_path)

# Load the sentence embedding model — used to convert user queries and text chunks into vector form
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load the local LLM (e.g., Mistral 7B in GGUF format) with context window size set to 2048 tokens
llm = Llama(model_path=mistral_path, n_ctx=2048)

# Initialize the PaddleOCR model with English language and angle classification enabled
ocr_model = PaddleOCR(use_angle_cls=True, lang='en')

In [7]:
# === Session Logs ===
cache = {}
chat_log = []

In [8]:
# === Cache ===

def load_cache():
    # Load cache from file if it exists
    return json.load(open(CACHE_FILE, "r", encoding="utf-8")) if os.path.exists(CACHE_FILE) else {}

def save_cache(cache):
    # Save cache to file as JSON
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=4, ensure_ascii=False)


In [9]:
def save_feedback(query, answer, source, feedback):
    # Prepare feedback data as a dictionary
    data = {"query": query, "answer": answer, "source": source, "feedback": feedback}
    
    # Check if feedback file already exists
    file_exists = os.path.isfile(FEEDBACK_FILE)
    
    # Open feedback file in append mode
    with open(FEEDBACK_FILE, "a", encoding="utf-8", newline='') as f:
        writer = csv.DictWriter(f, fieldnames=data.keys())
        
        # Write header if file is new
        if not file_exists:
            writer.writeheader()
        
        # Write feedback data to file
        writer.writerow(data)


In [10]:
# === Reset everything ===

def reset_all():
    """Clear cache, feedback log, temporary docs, and chat history."""
    # Clear cache file by writing empty JSON object
    open(CACHE_FILE, "w").write("{}")
    # Clear feedback log file with header row
    open(FEEDBACK_FILE, "w").write("query,answer,source,feedback\n")
    # Delete temporary documents folder, ignore errors if not present
    shutil.rmtree("temp_docs", ignore_errors=True)
    # Clear in-memory chat log
    chat_log.clear()
    # Return placeholders for UI components or variables to reset
    return [], None, None, None, None, None


In [11]:
# === OCR extraction ===

def extract_text_from_image(image_path):
    # Run OCR on image and get results
    ocr_result = ocr_model.ocr(image_path, cls=True)
    # Extract text lines from OCR output and join with newlines
    extracted_text = "\n".join([line[1][0] for block in ocr_result for line in block])
    return extracted_text


In [12]:
# === FAISS retrieval (+ optional OCR chunk) ===

# Perform semantic search over pre-embedded text chunks using a FAISS index
# Optionally, include extra text (e.g., OCR from an uploaded image) as an additional result
def search_top_k(query, k=2, extra_text=None):
    # Convert the query into an embedding vector using the sentence transformer
    embedding = embedding_model.encode([query])
    
    # Search the FAISS index to find the top-k most similar vectors (chunks)
    distances, indices = index.search(np.array(embedding).astype("float32"), k)
    
    # Retrieve the corresponding text chunks (with metadata) from the DataFrame
    faiss_results = df_chunks.iloc[indices[0]][["chunk_id", "filename", "chunk_text"]]
    
    # If additional text (like from OCR) is provided, add it as an extra row in the results
    if extra_text:
        extra_df = pd.DataFrame([{
            "chunk_id": "ocr_chunk",                # Identifier for the dynamic chunk
            "filename": "uploaded_image",           # Label the source as image input
            "chunk_text": extra_text                # Actual OCR-extracted text
        }])
        faiss_results = pd.concat([faiss_results, extra_df], ignore_index=True)

    # Return a DataFrame containing top-k retrieved chunks + optional extra text
    return faiss_results


In [13]:
# === Re-rank retrieved chunks using cosine similarity to refine semantic relevance ===

# Given a set of retrieved chunks, re-rank them by computing cosine similarity between
# the user's question and each chunk to improve semantic accuracy.
def rerank_chunks(chunks_df, question, model):
    # Generate the embedding vector for the input question
    q_vec = model.encode([question])[0]

    # Compute cosine similarity between the question vector and each chunk's embedding
    # Add a new column 'score' to store similarity values
    chunks_df["score"] = chunks_df["chunk_text"].apply(
        lambda text: cosine_similarity([q_vec], [model.encode([text])[0]])[0][0]
    )

    # Sort the chunks in descending order of similarity score and return the top 2
    return chunks_df.sort_values("score", ascending=False).head(2)


In [14]:
# === Best semantic snippet ===

# From a set of retrieved document chunks, identify the single sentence that is most semantically
# similar to the user's question, using cosine similarity on sentence embeddings.
def find_best_semantic_snippet(chunks_df, question, model, max_length=250):
    # Generate embedding for the user's question
    question_vec = model.encode([question])[0]

    # Initialize variables to track the best-matching sentence
    best_snippet, best_file, best_score = "", "", -1

    # Iterate through each chunk (row) in the DataFrame
    for _, row in chunks_df.iterrows():
        # Split chunk text into individual sentences
        sentences = re.split(r'(?<=[.!?]) +', row["chunk_text"])
        
        for sent in sentences:
            sent = sent.strip()
            
            # Skip very short sentences (likely noise)
            if len(sent) < 10:
                continue

            # Compute cosine similarity between question and this sentence
            score = cosine_similarity([question_vec], [model.encode([sent])[0]])[0][0]

            # Update best match if current sentence is more similar
            if score > best_score:
                best_score = score
                best_snippet = sent
                best_file = row["filename"]

    # If no suitable sentence was found, return a fallback (first chunk, truncated)
    if not best_snippet:
        return chunks_df.iloc[0]["filename"], chunks_df.iloc[0]["chunk_text"][:max_length].strip() + "..."

    # Return the best-matching filename and sentence, truncated if too long
    return best_file, best_snippet if len(best_snippet) < max_length else best_snippet[:max_length].strip() + "..."


In [15]:
# === Prompt & answer generation ===
# The prompt follows a structured format to guide the model in generating helpful answers.
def build_prompt(chunks, question, chat_log=None):
    # Concatenate the retrieved text chunks into a single context block
    context = "\n".join(chunks["chunk_text"].tolist())

    # Initialize conversation history string (if available)
    history_text = ""
    if chat_log:
        # Include only the last 2 rounds of Q&A to keep the prompt within context limits
        for i, entry in enumerate(chat_log[-2:], start=1):
            history_text += f"Q{i}: {entry['question']}\nA{i}: {entry['answer']}\n"

    # Compose the full prompt in a structured, instructive format
    prompt = (
        "You are a helpful assistant. Answer the user's question using the provided context If unsure, say 'I don't know'.\n\n"
        "### Prior Conversation ###\n" + history_text +
        "### Context from Documents ###\n" + context + "\n\n" +
        "### Question ###\n" + question + "\n\n" +
        "### Answer ###\n"
    )

    # Return the fully constructed prompt for LLM input
    return prompt


In [16]:
def generate_answer(chunks, question, chat_log=None):
    # Build prompt using document chunks, question, and chat history
    prompt = build_prompt(chunks, question, chat_log)
    
    try:
        # Try generating response using local LLM
        response = llm(prompt, max_tokens=200, stop=["</s>", "###"])
        return response["choices"][0]["text"].strip()
    
    except Exception as e:
        # If local LLM fails, fallback to OpenAI API (if configured)
        print("Local LLM failed. Using OpenAI fallback...", e)
        try:
            import openai
            openai.api_key = os.getenv("OPENAI_API_KEY")  # Load API key from environment
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=200
            )
            return response['choices'][0]['message']['content'].strip()
        
        except:
            # If both fail, return error message
            return "Sorry, both local and fallback LLMs failed."


In [17]:
query = "Who is Marieve Herington?"

# Step 1: Retrieve relevant chunks from FAISS index
relevant_chunks = search_top_k(query, k=3)

# Step 2: Generate answer using those chunks
answer = generate_answer(relevant_chunks, query, chat_log)

print("Answer:", answer)


llama_perf_context_print:        load time =  248242.45 ms
llama_perf_context_print: prompt eval time =  248227.58 ms /   988 tokens (  251.24 ms per token,     3.98 tokens per second)
llama_perf_context_print:        eval time =  103886.27 ms /   149 runs   (  697.22 ms per token,     1.43 tokens per second)
llama_perf_context_print:       total time =  352337.12 ms /  1137 tokens


Answer: Marieve Herington is a Canadian actress and jazz singer who has appeared in recurring roles on How I Met Your Mother, Your Mother, Good Luck Charlie, and Ever After High. She also provides the voice of Tilly Green on the Disney Channel show. She began singing at the age of 12 and has been fronting her own jazz ensembles since the age of 16. She studied drama at the University of Toronto and has appeared in numerous commercials and has joined the Alliance of Canadian Cinema Television and Radio Artists (ACTRA) at the age of 9. She has released several albums, including Blossoming and Midnight Sessions, and has sung the theme songs for several television shows.


In [19]:
# Handles user feedback submission by saving it and updating the UI history
def feedback_fn(query, answer, source, feedback, history):
    # Save the feedback to the feedback log (CSV file)
    save_feedback(query, answer, source, feedback)
    # Append a feedback confirmation message to the chat history
    return history + [(" Feedback received: " + feedback.upper(), "")]

# Retrieves the count of good/bad feedback for a given query from the feedback log
def get_feedback_score(query):
    # If the feedback file does not exist, return a default message
    if not os.path.exists(FEEDBACK_FILE):
        return "No feedback yet."
    # Load the feedback log into a DataFrame
    df = pd.read_csv(FEEDBACK_FILE)
    # Filter for rows that match the given query
    match = df[df["query"] == query]
    # If no matching feedback is found, return a default message
    if match.empty:
        return "No feedback yet."
    # Count the number of 'good' and 'bad' feedback entries
    good = match[match["feedback"] == "good"].shape[0]
    bad = match[match["feedback"] == "bad"].shape[0]
     # Return a summary of the feedback counts
    return f"Feedback: 👍 {good} | 👎 {bad}"


In [20]:
# Exports the current chat log to a CSV file with a timestamped filename
def export_chat_to_csv():
    from gradio import update  # Import update for dynamic UI changes

    # If chat_log is empty, hide the export download component
    if not chat_log:
        return update(value=None, visible=False)

    # Generate a timestamped filename for the export
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"chat_export_{ts}.csv"

    # Convert the chat_log (a list of tuples or dicts) into a DataFrame and save as CSV
    pd.DataFrame(chat_log).to_csv(path, index=False)

    # Make the export download component visible and set its file path
    return update(value=path, visible=True)

# Exports the feedback log CSV (if it exists) by copying it to a new timestamped file
def export_feedback_to_csv():
    from gradio import update  # Import update for dynamic UI changes

    # If feedback file does not exist, hide the export download component
    if not os.path.exists(FEEDBACK_FILE):
        return update(value=None, visible=False)

    # Generate a timestamped filename for the feedback export
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    new_path = f"feedback_export_{ts}.csv"

    # Copy the existing feedback file to the new path
    shutil.copy(FEEDBACK_FILE, new_path)

    # Make the export download component visible and set its file path
    return update(value=new_path, visible=True)


In [ ]:
# === Main chat function ===
def chatbot_ui(query, history, image_file):
    cache = load_cache()  # Load cached responses if any

    # Extract text from image if uploaded
    ocr_text = extract_text_from_image(image_file.name) if image_file else None

    # Check if the query is already in cache
    if query in cache:
        answer = cache[query]["answer"]
        source = cache[query]["source"]
        snippet = cache[query]["snippet"]
        filepath = os.path.join("temp_docs", source)  # Path to cached source file
    else:
        # Search top-k chunks from documents (includes OCR text if present)
        chunks_raw = search_top_k(query, k=5, extra_text=ocr_text)

        # Re-rank the chunks based on semantic similarity
        chunks = rerank_chunks(chunks_raw, query, embedding_model)

        # Generate an answer using the top-ranked chunks
        answer = generate_answer(chunks, query, chat_log)

        # Identify the best matching document snippet and source
        source, snippet = find_best_semantic_snippet(chunks, query, embedding_model)

        # Copy the source file to a temporary location for download
        local_file_path = os.path.join(text_folder, source)
        os.makedirs("temp_docs", exist_ok=True)  # Ensure temp_docs directory exists
        filepath = os.path.join("temp_docs", source)
        if os.path.exists(local_file_path):
            shutil.copy(local_file_path, filepath)
        else:
            filepath = None  # If file doesn't exist, disable file output

        # Cache the new query response
        cache[query] = {"answer": answer, "source": source, "snippet": snippet}
        save_cache(cache)

    # Add Q&A with source and snippet to chat log
    chat_log.append({"question": query, "answer": answer, "source": source, "snippet": snippet})
    feedback_summary = get_feedback_score(query)
    display = f" Answer: {answer}\n Source: {source}\n Snippet: {snippet}\n {feedback_summary}"

    # Add this round of interaction to chat history
    history.append((query, display))

    from gradio import update
    # Show download button for the source file if it exists, otherwise hide
    file_output = filepath if filepath and os.path.exists(filepath) else update(visible=False)

    # Return updated history and outputs
    return history, query, answer, source, file_output


In [21]:
# === Main chat function ===

def chatbot_ui(query, history, image_file):
    cache = load_cache()
    ocr_text = extract_text_from_image(image_file.name) if image_file else None

    if query in cache:
        answer = cache[query]["answer"]
        source = cache[query]["source"]
        snippet = cache[query]["snippet"]
        filepath = os.path.join("temp_docs", source)
    else:
        chunks_raw = search_top_k(query, k=5, extra_text=ocr_text)
        chunks = rerank_chunks(chunks_raw, query, embedding_model)
        answer = generate_answer(chunks, query, chat_log)
        source, snippet = find_best_semantic_snippet(chunks, query, embedding_model)

        local_file_path = os.path.join(text_folder, source)
        os.makedirs("temp_docs", exist_ok=True)
        filepath = os.path.join("temp_docs", source)
        if os.path.exists(local_file_path):
            shutil.copy(local_file_path, filepath)
        else:
            filepath = None

        cache[query] = {"answer": answer, "source": source, "snippet": snippet}
        save_cache(cache)

    chat_log.append({"question": query, "answer": answer, "source": source, "snippet": snippet})
    feedback_summary = get_feedback_score(query)
    display += f"\n {feedback_summary}"


    display = f" Answer: {answer}\n Source: {source}\n Snippet: {snippet}"
    history.append((query, display))

    from gradio import update
    file_output = filepath if filepath and os.path.exists(filepath) else update(visible=False)
    return history, query, answer, source, file_output

In [22]:
# === Feedback summary loader ===

def load_feedback_summary():
    if not os.path.exists(FEEDBACK_FILE):
        return pd.DataFrame(columns=["query", "answer", "source", "feedback"])
    return pd.read_csv(FEEDBACK_FILE)